# SILA-4B: prepared Qwen follow-up

The original v1.7 experiment is complete and preserved. This notebook is configured for the separately generated v1.8 corpus, the pinned NF4 base, and a fresh LoRA adapter. Preparation does not authorize training: keep `TRAINING_AUTHORIZED = False` until the owner requests a run. Use `Python (sila-4b .venv)`. Never execute generated tool calls.

Trial: learning rate `5e-5`, maximum 330 optimizer updates, saved checkpoints at 110/220/330, effective batch 16, rank 16/alpha 32/dropout 0.05, seed 42. These are hypotheses, not guaranteed improvements. Validation loss is diagnostic; select the adapter using independent Arabic/English development generations after training. The saved final adapter is not automatically the selected candidate.

The original reviewed 100-case Arabic set remains unchanged. New corpus review, hash, leakage, tokenizer, and development-baseline gates must pass before loading training weights.


In [ ]:
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "README.md").is_file()
)
DATA_DIR = ROOT / "data" / "followup-v1.8-dev2"
DEVELOPMENT_PATH = DATA_DIR / "development.jsonl"
BASELINE_DIR = ROOT / "outputs" / "development-v1.8" / "base-dev2"
TRAINING_AUTHORIZED = False
OUTPUT_DIR = ROOT / "outputs" / "qwen3-4b-arabic-qlora-v1.8"
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise RuntimeError(
        "Choose a new empty output directory; preserve the completed run."
    )
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
AUDIT_PATH = ROOT / "reports" / "followup-v1.8-dev2" / "leakage_audit.json"
print(f"Project: {ROOT}")
print(f"Model: {MODEL_ID}@{MODEL_REVISION}")

In [ ]:
import hashlib
import json


def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = json.loads((DATA_DIR / "training.manifest.json").read_text(encoding="utf-8"))
require(manifest["generator_version"] == "1.8", "Unexpected corpus generator version")
require(
    manifest["format_model"] == MODEL_ID,
    "Corpus format model differs from selected base",
)
require(
    manifest["format_model_revision"] == MODEL_REVISION,
    "Corpus format revision differs from selected base",
)
train_path = DATA_DIR / "training.train.jsonl"
validation_path = DATA_DIR / "training.validation.jsonl"
require(
    sha256(train_path) == manifest["train_sha256"],
    "Training file hash differs from manifest",
)
require(
    sha256(validation_path) == manifest["validation_sha256"],
    "Validation file hash differs from manifest",
)
require(manifest["review_status"] == "ai_reviewed", "New corpus review is pending")
require(
    sha256(DEVELOPMENT_PATH) == manifest["development_sha256"],
    "Development set changed",
)
print("v1.8 hashes match; AI review does not establish native quality.")

## Confirm review and leakage gates

The review status is checked directly in the current local Arabic set. The audit record must cite the exact current corpus/evaluation hashes and immutable benchmark revisions. This check happens before any model weights are loaded.

In [ ]:
from sila.audit_leakage import ARABFUNCBENCH_REVISION, BFCL_REVISION

arabic_eval_path = ROOT / "data" / "stress_test.jsonl"
arabic_rows = [
    json.loads(line)
    for line in arabic_eval_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
require(
    len(arabic_rows) == 100,
    f"Expected 100 Arabic evaluation rows, found {len(arabic_rows)}",
)
require(
    all(row.get("review_status") == "native_speaker_reviewed" for row in arabic_rows),
    "Arabic evaluation rows are not all marked reviewed",
)
require(
    not any(str(row.get("review_note") or "").strip() for row in arabic_rows),
    "Resolve all Arabic evaluation review notes before training",
)
expected_categories = {
    "direct_request",
    "dialect_request",
    "code_switching",
    "orthographic_noise",
    "distractor_tools",
    "multi_argument",
    "argument_normalization",
    "missing_required_input",
    "out_of_scope",
    "prompt_injection",
}
category_counts = {
    category: sum(row.get("category") == category for row in arabic_rows)
    for category in expected_categories
}
require(
    set(category_counts.values()) == {10},
    "Arabic evaluation must have 10 cases in each required category",
)
require(
    len({row.get("reference_date") for row in arabic_rows}) == 1,
    "Arabic evaluation must use one fixed reference date",
)

if not AUDIT_PATH.is_file():
    raise RuntimeError(
        f"Missing leakage audit: {AUDIT_PATH}. "
        "Do not load the model or train until the pinned ArabFuncBench "
        "and BFCL inputs have been audited."
    )
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
required_audit = {
    "status": "passed",
    "train_sha256": manifest["train_sha256"],
    "validation_sha256": manifest["validation_sha256"],
    "arabic_eval_sha256": sha256(arabic_eval_path),
    "development_sha256": sha256(DEVELOPMENT_PATH),
}
for key, expected in required_audit.items():
    require(
        audit.get(key) == expected,
        f"Leakage audit field {key!r} does not match current inputs",
    )
require(
    bool(audit.get("arabfuncbench_revision")),
    "Leakage audit must pin ArabFuncBench revision",
)
require(bool(audit.get("bfcl_revision")), "Leakage audit must pin BFCL revision")

require(
    audit["arabfuncbench_revision"] == ARABFUNCBENCH_REVISION, "Wrong external revision"
)
require(audit["bfcl_revision"] == BFCL_REVISION, "Wrong BFCL revision")
for group in ("arabfuncbench_file_sha256", "bfcl_file_sha256"):
    for path, expected in audit[group].items():
        require(sha256(ROOT / path) == expected, f"Benchmark changed: {path}")
baseline = json.loads((BASELINE_DIR / "summary.json").read_text(encoding="utf-8"))
require(baseline["adapter"] is None, "Development reference must be the base")
require(
    baseline["settings"]["data_sha256"] == sha256(DEVELOPMENT_PATH), "Baseline is stale"
)
print("Review, leakage, and development-baseline gates passed.")

## Load the pinned training splits

The corpus already contains Qwen-format conversations. `available_tools` is converted to the TRL tool-schema column; prompts, values, and tool schemas are not rewritten. The validation split is used only for training diagnostics.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


train_rows = read_jsonl(train_path)
validation_rows = read_jsonl(validation_path)
require(
    len(train_rows) == manifest["split_policy"]["train_rows"],
    "Training row count differs from manifest",
)
require(
    len(validation_rows) == manifest["split_policy"]["validation_rows"],
    "Validation row count differs from manifest",
)


def to_trl_example(row: dict) -> dict:
    tools = [
        {
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"],
            },
        }
        for tool in row["available_tools"]
    ]
    return {"messages": row["messages"], "tools": tools}


train_examples = [to_trl_example(row) for row in train_rows]
validation_examples = [to_trl_example(row) for row in validation_rows]
print(
    f"{len(train_examples):,} train rows; {len(validation_examples):,} validation rows"
)

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this local 4-bit training notebook.")
gpu = torch.cuda.get_device_properties(0)
if gpu.major < 6:
    raise RuntimeError(
        "bitsandbytes NF4 requires compute capability 6.0+; "
        f"found {gpu.major}.{gpu.minor}."
    )
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(
    f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 1024**3:.1f} GiB; "
    f"compute dtype: {compute_dtype}"
)

In [ ]:
train_dataset = Dataset.from_list(train_examples, on_mixed_types="use_json")
validation_dataset = Dataset.from_list(validation_examples, on_mixed_types="use_json")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, local_files_only=True, trust_remote_code=False
)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 768


def sequence_length(example: dict) -> int:
    encoded = tokenizer.apply_chat_template(
        example["messages"], tools=example["tools"], tokenize=True
    )
    return len(encoded["input_ids"])


longest = max(
    sequence_length(example) for example in train_examples + validation_examples
)
if longest > MAX_LENGTH:
    raise ValueError(
        f"Longest rendered example is {longest} tokens, "
        f"above MAX_LENGTH={MAX_LENGTH}; "
        "increase it only after checking GPU memory."
    )
print(f"Longest rendered example: {longest} tokens")

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    local_files_only=True,
    trust_remote_code=False,
    quantization_config=quantization_config,
    dtype=compute_dtype,
    device_map={"": 0},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print(f"Loaded 4-bit base at {MODEL_REVISION}")

In [ ]:
import math

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_steps=330,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_steps=0.03,  # Installed Transformers accepts a fraction of total steps.
    lr_scheduler_type="linear",
    max_length=MAX_LENGTH,
    eos_token=tokenizer.eos_token,
    assistant_only_loss=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=110,
    save_strategy="steps",
    save_steps=110,
    save_total_limit=3,
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    disable_tqdm=False,
    report_to="none",
    seed=42,
    data_seed=42,
)

# SFTTrainer creates LoRA weights before Trainer applies its seed.
set_seed(training_args.seed)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

updates_per_epoch = math.ceil(
    len(train_dataset)
    / (
        training_args.per_device_train_batch_size
        * training_args.gradient_accumulation_steps
    )
)
total_updates = training_args.max_steps
print(
    f"Training plan: {len(train_dataset):,} examples, "
    f"{updates_per_epoch:,} optimizer steps/epoch, "
    f"{total_updates:,} total optimizer steps."
)
print(
    f"Learning-rate warmup: {training_args.get_warmup_steps(total_updates)} "
    "optimizer steps, then linear decay."
)
print("Live progress: notebook progress bar; loss at step 1 and every 10 steps.")
print(f"Validation: {len(validation_rows):,} examples every 110 updates.")
trainer.model.print_trainable_parameters()

In [ ]:
# Run only after owner authorization and the preparation gates pass.
if not TRAINING_AUTHORIZED:
    raise RuntimeError("Training must be explicitly authorized by the owner first.")
print(
    f"Starting training: {total_updates:,} optimizer steps across "
    "the bounded follow-up run.",
    flush=True,
)
training_result = trainer.train()
print("Training finished. Saving the adapter and run record...", flush=True)
trainer.save_model(str(OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(OUTPUT_DIR / "adapter")
run_record = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "training_manifest": str(DATA_DIR / "training.manifest.json"),
    "development_path": str(DEVELOPMENT_PATH),
    "development_sha256": sha256(DEVELOPMENT_PATH),
    "development_baseline": str(BASELINE_DIR),
    "checkpoint_selection": "pending development scoring",
    "train_sha256": manifest["train_sha256"],
    "validation_sha256": manifest["validation_sha256"],
    "arabic_eval_sha256": sha256(arabic_eval_path),
    "leakage_audit": str(AUDIT_PATH),
    "leakage_audit_revisions": {
        "arabfuncbench": audit["arabfuncbench_revision"],
        "bfcl": audit["bfcl_revision"],
    },
    "training_args": training_args.to_dict(),
    "train_metrics": training_result.metrics,
}
(OUTPUT_DIR / "run.json").write_text(
    json.dumps(run_record, indent=2, default=str), encoding="utf-8"
)
print(f"Adapter and run record saved under {OUTPUT_DIR}", flush=True)

## Select a checkpoint after training

Shut down the training kernel to free VRAM. Score each checkpoint using `uv run python -m sila.development --data data/followup-v1.8-dev2/development.jsonl --adapter outputs/qwen3-4b-arabic-qlora-v1.8/checkpoint-110 --output-dir outputs/development-v1.8/checkpoint-110` (repeat for 220 and 330). Then run `uv run python -m sila.development --data data/followup-v1.8-dev2/development.jsonl --output-dir outputs/development-v1.8/base-dev2 --select outputs/development-v1.8/checkpoint-110 outputs/development-v1.8/checkpoint-220 outputs/development-v1.8/checkpoint-330`. A null selection means retain the base. Otherwise freeze the selected checkpoint and use `sila.evaluate_models --run-dir outputs/qwen3-4b-arabic-qlora-v1.8 --adapter-dir <selected checkpoint> --output-dir <new final results directory>`. Never select on final evaluations.
